# SysCraft WARA-PS MQTT Packet Test

Use this notebook in JupyterLab to test both packet types consumed by Arena: one retained `SensorInfo` snapshot and compact `HeartBeat` liveness packets.

In [ ]:
# Run once if needed:
# %pip install paho-mqtt

## Arena connection

Configure the Arena UI with root topic `syscraft/arena`; it subscribes to `syscraft/arena/#`. This notebook publishes:

```text
syscraft/arena/unit/air/simulation/mqtt-contract-test-01/sensor_info
syscraft/arena/unit/air/simulation/mqtt-contract-test-01/heartbeat
```

In [ ]:
import getpass
import json
import os
import threading
import time
import uuid

import paho.mqtt.client as mqtt

MQTT_HOST = os.getenv("SYSCRAFT_MQTT_HOST", "mqtt.conceptio.cloud")
MQTT_PORT = int(os.getenv("SYSCRAFT_MQTT_PORT", "1883"))
MQTT_USERNAME = os.getenv("SYSCRAFT_MQTT_USERNAME", "chris")
MQTT_PASSWORD = os.getenv("SYSCRAFT_MQTT_PASSWORD")
# Use a distinct ID so this notebook does not disconnect another publisher.
# Set SYSCRAFT_MQTT_CLIENT_ID=chris only if your broker explicitly requires it.
MQTT_CLIENT_ID = os.getenv("SYSCRAFT_MQTT_CLIENT_ID", "syscraft-mqtt-contract-test")
MQTT_ROOT_TOPIC = os.getenv("SYSCRAFT_MQTT_ROOT_TOPIC", "syscraft/arena")

UNIT = "mqtt-contract-test-01"
AGENT_TYPE = "air"  # WARA-PS: air, ground, surface, or subsurface
# The renderer uses SIDC to choose the icon; domain alone does not change it.
DEFAULT_SIDC_BY_AGENT_TYPE = {
    "air": "130301000011010000000000000000",
    "ground": "130315000016040000000000000000",
    "surface": "130330000012050100000000000000",
    "subsurface": "130303000011010000000000000000",
}
ENTITY_SIDC = DEFAULT_SIDC_BY_AGENT_TYPE[AGENT_TYPE]

def topic(endpoint):
    return f"{MQTT_ROOT_TOPIC.strip('/')}/unit/{AGENT_TYPE}/simulation/{UNIT}/{endpoint}"

SENSOR_INFO_TOPIC = topic("sensor_info")
HEARTBEAT_TOPIC = topic("heartbeat")
print(f"Broker: {MQTT_HOST}:{MQTT_PORT}")
print(f"SensorInfo: {SENSOR_INFO_TOPIC}")
print(f"HeartBeat: {HEARTBEAT_TOPIC}")

In [ ]:
publisher = {"client": None, "thread": None, "stop": threading.Event(), "agent_uuid": str(uuid.uuid4())}

def sensor_info_packet(snapshot_interval_seconds=30):
    return {
        "name": UNIT,
        "rate": 1 / snapshot_interval_seconds,
        "sensor-data-provided": ["position", "heading"],
        "stamp": time.time(),
        "type": "SensorInfo",
        # SysCraft extension: full map data is sent only in this low-rate packet.
        "arena-entity": {
            "id": UNIT,
            "displayName": "MQTT Contract Test",
            "domain": AGENT_TYPE,
            "assetType": "vehicle",
            "systemType": "mqtt-contract-test",
            "latitude": 29.1886,
            "longitude": -81.0487,
            "altitudeMeters": 350,
            "headingDegrees": 90,
            "sidc": ENTITY_SIDC,
            "affiliation": "friend",
            "renderMode": "mil-symbol",
            "symbology": {"uniqueDesignation": "MQTT TEST"},
        },
    }

def heartbeat_packet(heartbeat_interval_seconds=1):
    return {
        "agent-type": AGENT_TYPE,
        "agent-uuid": publisher["agent_uuid"],
        "levels": ["sensor"],
        "name": UNIT,
        "rate": 1 / heartbeat_interval_seconds,
        "stamp": time.time(),
        "type": "HeartBeat",
    }

def packet_json(packet):
    return json.dumps(packet, separators=(",", ":"))

def preview_packets():
    """Inspect the exact two packets without connecting to MQTT."""
    print(f"{SENSOR_INFO_TOPIC}\n{packet_json(sensor_info_packet())}")
    print(f"{HEARTBEAT_TOPIC}\n{packet_json(heartbeat_packet())}")

## Preview before publishing

Run this cell to inspect both JSON messages. It does not connect to the broker.

In [ ]:
preview_packets()

In [ ]:
def _publish_loop(heartbeat_interval_seconds=1, snapshot_interval_seconds=30):
    password = MQTT_PASSWORD or getpass.getpass("MQTT password: ")
    connected = threading.Event()
    connection_error = []
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=MQTT_CLIENT_ID, protocol=mqtt.MQTTv5)
    client.username_pw_set(MQTT_USERNAME, password)

    def on_connect(_client, _userdata, _flags, reason_code, _properties):
        if reason_code.is_failure:
            connection_error.append(f"Broker rejected connection: {reason_code}")
        else:
            print(f"Connected to {MQTT_HOST}:{MQTT_PORT} as {MQTT_CLIENT_ID}")
            connected.set()

    def on_disconnect(_client, _userdata, _disconnect_flags, reason_code, _properties):
        if not publisher["stop"].is_set():
            connection_error.append(f"Disconnected from broker: {reason_code}")
            print(connection_error[-1])

    client.on_connect = on_connect
    client.on_disconnect = on_disconnect
    try:
        client.connect(MQTT_HOST, MQTT_PORT, 60)
        client.loop_start()
        publisher["client"] = client
        if not connected.wait(timeout=10):
            reason = connection_error[-1] if connection_error else "No CONNACK received within 10 seconds."
            raise ConnectionError(reason)

        last_snapshot_at = 0.0
        while not publisher["stop"].is_set():
            now = time.monotonic()
            if now - last_snapshot_at >= snapshot_interval_seconds:
                result = client.publish(SENSOR_INFO_TOPIC, packet_json(sensor_info_packet(snapshot_interval_seconds)), qos=1, retain=True)
                if result.rc != mqtt.MQTT_ERR_SUCCESS:
                    raise ConnectionError(f"SensorInfo publish failed: rc={result.rc}")
                result.wait_for_publish()
                print("SensorInfo published (retained)")
                last_snapshot_at = now
            result = client.publish(HEARTBEAT_TOPIC, packet_json(heartbeat_packet(heartbeat_interval_seconds)), qos=0, retain=False)
            if result.rc != mqtt.MQTT_ERR_SUCCESS:
                raise ConnectionError(f"HeartBeat publish failed: rc={result.rc}")
            print("HeartBeat published")
            time.sleep(heartbeat_interval_seconds)
    except Exception as error:
        print(f"Publisher error: {error}")
    finally:
        client.loop_stop()
        client.disconnect()
        publisher["client"] = None
        print("Publisher stopped.")

def start_publisher(heartbeat_interval_seconds=1, snapshot_interval_seconds=30):
    if heartbeat_interval_seconds <= 0 or snapshot_interval_seconds <= 0:
        raise ValueError("Intervals must be greater than zero.")
    if publisher["thread"] and publisher["thread"].is_alive():
        print("Publisher is already running.")
        return
    publisher["stop"].clear()
    publisher["agent_uuid"] = str(uuid.uuid4())
    publisher["thread"] = threading.Thread(target=_publish_loop, args=(heartbeat_interval_seconds, snapshot_interval_seconds), daemon=True)
    publisher["thread"].start()
    print("Publisher starting...")

def stop_publisher():
    publisher["stop"].set()
    if publisher["thread"]:
        publisher["thread"].join(timeout=5)
    print("Stop requested.")

## Start test

This immediately publishes the retained snapshot, then sends a heartbeat every second. The Arena should display `MQTT Contract Test` after the snapshot arrives.

In [ ]:
start_publisher(heartbeat_interval_seconds=1, snapshot_interval_seconds=30)

## Stop test

In [ ]:
stop_publisher()